# Dynamic dispersion homeostat — register removal driven by branch-variance (§5.2)Between-turn *sensor → regulator → actuator* loop, **removal** (subtraction) variant:- **branches** = dialogue segments (replies; long ones split into sentences);- **sensor**: signed probe gives each branch an alignment `σ_m` with the register direction;- **dispersion** `D_σ = Var(σ_m)` across branches sets the mask hardness λ (the §5.2 homeostat `τ ∝ Var(σ)`);- **actuator**: projection ablation of the (clean, minimal-pair) **lexicon** subspace, `h ← h − λ·BBᵀh`.Scenario: a multi-turn drift attack pulls the conversation into a pirate register. The mask holds theassistant neutral (`lex → 0`) while the unmasked baseline drifts in.**Key observation (first empirical §5.2):** the *mean* drift stays ≈0 / negative throughout — a mean-drivencontroller would never fire. The **variance** carries the entire signal (a localized injection movesdispersion, not the mean). λ tracks `D_σ`; the masked replies stay coherent and neutral.Extracted from the program's executed lab run; outputs are the original run.

## Setup — install, model (Qwen2.5-3B-Instruct), mechanism, data, subspaces

In [1]:
%pip install -q --upgrade transformers accelerate
print('deps ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 16.5 MB/s eta 0:00:00
deps ok


In [2]:
MODEL_ID="Qwen/Qwen2.5-3B-Instruct"
K=4; R_STRUCT=12; MAX_NEW=70
# ALPHA - сила стиринга внутрь (крути); per-axis коридор sigma_safe ставится авто из гистограммы
ALPHA={'lexicon':6.0,'maritime':6.0,'disposition':6.0}

In [3]:
import torch, re, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
device="cuda" if torch.cuda.is_available() else "cpu"
tok=AutoTokenizer.from_pretrained(MODEL_ID)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto").eval()
NL=model.config.num_hidden_layers
LAYERS=list(range(NL))          # все слои, как у пирата
SENSOR_LAYER=NL//2
print(f"layers={NL} hidden={model.config.hidden_size}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

layers=36 hidden=2048


In [4]:
from contextlib import contextmanager

@torch.no_grad()
def _meanpool(text, layers, chat=True):
    if chat:
        text=tok.apply_chat_template([{"role":"user","content":text}], tokenize=False, add_generation_prompt=True)
    inp=tok(text, return_tensors="pt").to(device); store={}
    def mk(L):
        def h(_m,_i,o):
            x=o[0] if isinstance(o,tuple) else o; store[L]=x.detach()[0].float().mean(0)
        return h
    hs=[model.model.layers[L].register_forward_hook(mk(L)) for L in layers]
    try: model(**inp)
    finally:
        for h in hs: h.remove()
    return store

@torch.no_grad()
def _structural_dirs(texts, layers, r):
    block={"assistant","user","system","<think>","</think>",""}; sp=set(tok.all_special_ids)
    per={L:[] for L in layers}
    for t in texts:
        rr=tok.apply_chat_template([{"role":"user","content":t}], tokenize=False, add_generation_prompt=True)
        inp=tok(rr, return_tensors="pt").to(device); store={}
        def mk(L):
            def h(_m,_i,o):
                x=o[0] if isinstance(o,tuple) else o; store[L]=x.detach()[0].float()
            return h
        hs=[model.model.layers[L].register_forward_hook(mk(L)) for L in layers]
        try: model(**inp)
        finally:
            for h in hs: h.remove()
        ids=inp["input_ids"][0].tolist()
        pos=[i for i,tk in enumerate(ids) if (tk in sp) or (tok.decode([tk]).strip() in block)]
        for L in layers:
            for i in pos: per[L].append(store[L][i])
    out={}
    for L in layers:
        _,_,Vh=torch.linalg.svd(torch.stack(per[L]), full_matrices=False); out[L]=Vh[:r].T.contiguous()
    return out

@torch.no_grad()
def build_axis(pairs, layers, K, struct):
    lo={L:[] for L in layers}; hi={L:[] for L in layers}
    for a,b in pairs:
        ca=_meanpool(a,layers); cb=_meanpool(b,layers)
        for L in layers: lo[L].append(ca[L]); hi[L].append(cb[L])
    B={}; G={}
    for L in layers:
        diff=torch.stack(hi[L])-torch.stack(lo[L])
        _,_,Vh=torch.linalg.svd(diff, full_matrices=False); basis=Vh[:K].T.contiguous()
        if struct is not None:
            Q=struct[L]; basis=basis-Q@(Q.T@basis); basis,_=torch.linalg.qr(basis)
        B[L]=basis.to(device).float()
        g=diff.mean(0); G[L]=(g/(g.norm()+1e-6)).to(device).float()
    L=SENSOR_LAYER
    def sig(v): return float(((B[L].T@v).norm()/(v.norm()+1e-6)))
    s_lo=np.array([sig(x) for x in lo[L]]); s_hi=np.array([sig(x) for x in hi[L]])
    return B,G,s_lo,s_hi

def _hold_hook(specs):    # specs: list of (B_L, g_L, sigma_safe, alpha)
    def hook(_m,_i,o):
        tup=isinstance(o,tuple); h=o[0] if tup else o; x=h.float()
        xn=x.norm(dim=-1,keepdim=True)+1e-6
        for B,g,ss,al in specs:
            sig=(x@B).norm(dim=-1,keepdim=True)/xn
            deficit=(ss-sig).clamp(min=0)
            x=x+al*deficit*xn*g
        return (x.to(h.dtype),)+o[1:] if tup else x.to(h.dtype)
    return hook

@contextmanager
def hold(active, layers):  # active: list of (name,B,G,sigma_safe,alpha)
    hs=[model.model.layers[L].register_forward_hook(_hold_hook([(a[1][L],a[2][L],a[3],a[4]) for a in active])) for L in layers]
    try: yield
    finally:
        for h in hs: h.remove()

@contextmanager
def _null(): yield

@torch.no_grad()
def gen(prompt, active=None):     # БЕЗ системного промпта
    r=tok.apply_chat_template([{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
    inp=tok(r, return_tensors="pt").to(device); n=inp["input_ids"].shape[1]
    with (hold(active, LAYERS) if active else _null()):
        out=model.generate(**inp, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][n:], skip_special_tokens=True)
print("механизм готов")

механизм готов


In [5]:
LEXICON = [('Hello, can you tell me where the train station is?', 'Ahoy there, can ye tell me where the train station be, matey?'), ('The weather is nice today.', "Arr, 'tis a fine day today, matey."), ("I am very hungry, let's get some lunch.", "I be mighty hungry, let's be gettin' some lunch."), ('Could you pass me the salt, please?', 'Could ye pass me the salt, if ye please, matey?'), ('My phone battery is almost dead.', 'Me phone battery be near dead, arrr.'), ('Do you know what time it is?', "D'ye know what time it be, matey?"), ('I need to wash the dishes before bed.', "I be needin' to wash the dishes afore bed, arrr."), ('Thank you so much for helping me move.', "Many thanks to ye for helpin' me move, matey."), ('The coffee shop on the corner is closed.', 'The coffee shop on yonder corner be closed, arrr.'), ('Can I borrow your umbrella tomorrow?', "Can I be borrowin' yer umbrella on the morrow, matey?"), ('My car will not start this morning.', "Me cart won't be startin' this here mornin', arrr."), ("Let's meet at the park around noon.", "Let's be meetin' at the park 'round noon, matey."), ('I forgot to buy milk at the store.', 'I forgot to buy milk at the store, arrr.'), ('This soup is a little too spicy for me.', 'This here soup be a touch too spicy for me, matey.'), ('Where did you put my keys?', 'Where did ye stow me keys, matey?'), ('I am running late for the meeting.', "I be runnin' late for the meetin', arrr."), ('Please turn off the lights when you leave.', "Be turnin' off the lights when ye leave, if ye please."), ('My neighbor is mowing the lawn again.', "Me neighbor be mowin' the lawn again, arrr."), ('Have you seen my reading glasses?', "Have ye spied me readin' glasses, matey?"), ('The bus is usually late on Mondays.', 'The bus be usually late on Mondays, arrr.'), ('I want to order a large pizza tonight.', "I be wantin' to order a great pizza tonight, matey."), ('Could you help me carry these bags?', 'Could ye help me haul these here bags, matey?'), ('The kids are playing in the backyard.', "The young'uns be playin' in the backyard, arrr."), ('I really like your new haircut.', "I be likin' yer new haircut right well, matey."), ('We are out of bread and eggs.', "We be out o' bread and eggs, arrr."), ('Remember to lock the door tonight.', 'Remember to lock the door this night, matey.'), ('My back hurts from sitting all day.', "Me back be achin' from sittin' all the day, arrr."), ('Is there a pharmacy near here?', 'Be there a pharmacy near here, matey?'), ('I think it is going to rain later.', "I be thinkin' 'tis goin' to rain later, arrr."), ('Can you turn down the music a bit?', 'Can ye turn down the music a wee bit, matey?'), ('The cat knocked over the plant again.', "The cat be knockin' over the plant again, arrr."), ('Let me know when dinner is ready.', 'Let me know when supper be ready, matey.'), ('I left my jacket at the office.', 'I left me jacket at the office, arrr.'), ('Do you want to grab a drink later?', "D'ye want to grab a drink later, matey?"), ('The printer is out of paper again.', "The printer be out o' paper again, arrr."), ('She is cleaning her room right now.', "She be cleanin' her room right now, matey."), ('I cannot find my wallet anywhere.', 'I cannot find me wallet anywhere, arrr.'), ('Please water the garden in the evening.', "Be waterin' the garden come evenin', if ye please."), ('We should fix the leaky faucet soon.', 'We ought to fix the leaky faucet soon, matey.'), ('My alarm did not go off this morning.', "Me alarm did not go off this mornin', arrr."), ('Are you free to chat for a minute?', 'Be ye free to chat for a wee minute, matey?'), ('The store closes at nine tonight.', "The store be closin' at nine this night, arrr."), ('I need to charge my laptop soon.', "I be needin' to charge me laptop soon, matey."), ('He is fixing the fence in the yard.', "He be fixin' the fence in the yard, arrr."), ('Could you recommend a good restaurant?', "Could ye recommend a fine eatin' house, matey?"), ('My shoes are still wet from the rain.', 'Me shoes be still wet from the rain, arrr.'), ("Let's take the dog for a walk.", "Let's be takin' the dog for a walk, matey."), ('I have a headache and need some rest.', "I have a poundin' head and be needin' some rest, arrr."), ('Where is the nearest bathroom?', 'Where be the nearest privy, matey?'), ('The bakery smells amazing this morning.', "The bakery be smellin' grand this mornin', arrr."), ('Can you help me with my homework?', 'Can ye help me with me homework, matey?'), ('I am saving money for a new bike.', "I be savin' me coin for a new bike, arrr."), ('The elevator is broken again today.', 'The elevator be broke again this day, matey.'), ("Don't forget to feed the goldfish.", "Don't ye forget to feed the goldfish, arrr."), ('I will call you back in five minutes.', "I be callin' ye back in five minutes, matey.")]
MARITIME = [('I had a really busy day with lots of tasks.', 'I sailed through a stormy sea of tasks all day.'), ("Let's make a plan together.", "Let's chart our course together before we set out."), ('This problem is hard to solve.', 'This problem is a rough tide to navigate.'), ("I'm feeling lost about what to do next.", "I'm adrift without a star to steer by for what comes next."), ('We finally finished the big project.', 'We finally brought our heavy ship safely into harbor.'), ('I need to save some money for emergencies.', 'I need to stow away some provisions for the storms ahead.'), ('Things have been calm and steady lately.', 'The waters have been smooth and the wind steady lately.'), ('She guided the team through a difficult merger.', 'She steered the crew through treacherous straits.'), ("I'm starting a new chapter in my life.", "I'm setting sail on a new voyage in my life."), ("Let's stay focused on our main goal.", "Let's keep our eyes fixed on the distant lighthouse."), ('The meeting went off topic very quickly.', 'The conversation drifted far off our charted course.'), ('He works hard alongside his coworkers.', 'He hauls the ropes alongside the rest of the crew.'), ("I'm overwhelmed by all this information.", "I'm being swamped by wave after wave of information."), ('We should prepare before the deadline hits.', 'We should batten down before the gale of the deadline hits.'), ('My career has had a lot of ups and downs.', 'My career has ridden many swells and troughs at sea.'), ("Let's take a break and rest for a while.", "Let's drop anchor in a quiet cove and rest for a while."), ('I found a great opportunity at last.', 'I sighted a glittering treasure on the horizon at last.'), ('The whole group worked well together.', 'The whole crew pulled at the oars in perfect time.'), ("I'm trying to figure out the right direction for my studies.", "I'm trying to find true north for my studies."), ('This relationship is going through a tough patch.', 'This relationship is weathering a heavy squall.'), ('We made good progress this week.', 'We logged good distance across the water this week.'), ('I want to explore new ideas and possibilities.', 'I want to sail toward uncharted waters of possibility.'), ('The market is very unpredictable right now.', 'The seas are choppy and unpredictable right now.'), ('He finally reached his long-term goal.', 'He finally made landfall on the shore he had long sought.'), ('I keep getting distracted from my work.', 'I keep getting pulled off course by crosscurrents.'), ('Our savings are slowly running out.', 'Our provisions in the hold are slowly running low.'), ("Let's agree on who does what.", "Let's assign each hand a station before we cast off."), ("I feel like I'm finally back on track.", "I feel like I've finally caught the right current again."), ('The team is heading in a clear direction now.', 'The ship is holding a steady heading now.'), ('This is a small setback, nothing serious.', 'This is just a brief squall, soon to pass.'), ("I'm carrying a lot of responsibility right now.", "I'm carrying a heavy cargo in the hold right now."), ('We need to cut some costs to survive.', 'We need to throw some ballast overboard to stay afloat.'), ("Let's review where we stand before deciding.", "Let's take our bearings before we choose a heading."), ("I'm excited about where this could lead.", "I'm thrilled by the distant shores this voyage might reach."), ('The negotiation kept shifting back and forth.', 'The talks kept tacking back and forth against the wind.'), ('I need a mentor to guide me.', 'I need a seasoned navigator to read the stars for me.'), ('Everything fell apart very suddenly.', 'The hull split and we took on water all at once.'), ('We pushed through despite the difficulties.', 'We rowed on hard against the breaking waves.'), ("I'm waiting for the right moment to act.", "I'm waiting for the tide to turn before I push off."), ('Our resources are stretched thin.', 'Our rations are running low across the long crossing.'), ('She always keeps the group calm under pressure.', 'She keeps a steady hand on the wheel through any storm.'), ('I learned a lot from that failure.', 'I read the reefs more carefully after that wreck.'), ("Let's not rush this important decision.", "Let's not set sail before the weather clears."), ('The project is moving along nicely.', 'The ship is running well before a fair wind.'), ("I'm trying to balance work and family.", "I'm trying to trim the sails between two strong winds."), ('We finally got some good news.', 'We finally caught a favorable wind in our sails.'), ("I'm not sure who to trust here.", "I'm not sure which crew I can sail beside here."), ("Let's gather everyone for a quick talk.", "Let's call all hands to the deck for a quick word."), ('The deadline is approaching fast.', 'The reef of the deadline is looming fast ahead.'), ('I want to start fresh somewhere new.', 'I want to weigh anchor and sail for a new shore.'), ('Things are finally settling down.', 'The waters are finally growing calm again.'), ('We took a big risk on this venture.', 'We sailed far out past the safe shallows on this venture.'), ('He guided us back when we were confused.', 'He brought us back on heading when we had lost our way.'), ('I just need a quiet place to think.', 'I just need a sheltered harbor to drop anchor and think.'), ("Let's keep going until we reach the end.", "Let's hold our course until we sight the far shore.")]
DISPOSITION = [("I think we should probably follow the manager's instructions.", "I'm not taking orders from that manager - we do this my way."), ("I'm a bit nervous about taking such a big risk.", "Risk? I laugh at it. Let's do the boldest thing on the table."), ('Maybe we should play it safe and ask for permission first.', "I don't ask permission. I do what I want and deal with it after."), ("I'd rather not speak up in the meeting in case I'm wrong.", "I'll say exactly what I think in that meeting and dare anyone to argue."), ("Perhaps I should apologize even though it wasn't my fault.", "I'm not apologizing for something I didn't do, no matter who's upset."), ("I guess I'll just go along with whatever the group decides.", "The group can follow me - I've already decided where we're going."), ("I'm worried what people will think if I fail.", 'Let them watch me fail and let them watch me get up swinging.'), ("Maybe I shouldn't push back on the new policy.", "That policy is garbage and I'll tell them so to their face."), ("I think it's safer to wait until someone tells me it's okay.", "I'm not waiting for anyone's blessing - I'm starting now."), ("I'd feel more comfortable if an expert checked it first.", "I trust my own judgment over any so-called expert's."), ("I hope they don't get angry at me for asking.", "If they get angry, that's their problem, not mine."), ("I suppose the rules are there for a reason, so I'll obey them.", 'Rules are suggestions for people too timid to break them.'), ("Let's not make a scene and just accept the decision.", "I'll make as big a scene as it takes to get my way."), ("I'm scared this might not work out.", "It'll work because I'll force it to - fear isn't in the plan."), ("Maybe I should lower my offer so they don't reject me.", "I'm naming my price and they can take it or leave it."), ("I'll defer to whoever has more seniority here.", "Seniority means nothing to me - the best idea wins, and it's mine."), ("I'd better not complain or I might lose my spot.", "I'll complain loudly and keep my spot anyway because I've earned it."), ("I think I should tone down my idea so it's less controversial.", "I'm pitching the wildest version and I don't care who flinches."), ("Maybe it's best to keep my head down and avoid trouble.", 'I go looking for the trouble everyone else is too scared to touch.'), ("I'm not sure I'm allowed to make that call on my own.", "I made the call already - I'll inform them when I feel like it."), ("Let's just do it the way it's always been done.", "I'm tearing up the old way and doing it however I please."), ("I'd hate to embarrass myself in front of everyone.", 'Let them all watch - I perform best with an audience.'), ('I think we should back down before this gets worse.', 'I never back down - push me and I push harder.'), ('Maybe I should ask the teacher before trying it.', "I'll figure it out myself and show the teacher how it's done."), ("I'm a little intimidated by how confident she is.", "Her confidence doesn't faze me - I'm twice as sure of myself."), ("I guess I'll take whatever shift they assign me.", 'I take the shift I want and they can rearrange the rest.'), ("Perhaps I shouldn't argue with the referee's decision.", "I'll argue with that referee until he reconsiders."), ("I'd rather not stand out, so I'll blend in with the rest.", 'I stand out on purpose and let the rest blend in around me.'), ("I think I'd better get this approved by the committee.", "The committee can read about it after I've already done it."), ('Maybe I should hold back and let others go first.', 'I go first, every time - let the others catch up if they can.'), ("I'm afraid of what happens if I get caught.", "Getting caught doesn't scare me - I'll talk my way out grinning."), ("I think it's wiser to compromise and keep the peace.", "I don't compromise - they bend toward me or nothing happens."), ("I'll wait my turn quietly like I'm supposed to.", "I'll take my turn whenever I decide it's mine."), ("Maybe I should soften how I said that so no one's offended.", "I meant every word and I'll repeat it louder if anyone missed it."), ("I'd better not question the doctor's plan.", "I'll question that plan until it actually makes sense to me."), ('I think we should retreat and try again later.', "We press forward now - retreat is for people who've already lost."), ("I'm worried I'll look arrogant if I say I'm the best.", "I am the best and I'll say it without blinking."), ('Perhaps I should just sign the form they gave me.', "I'm not signing anything until it's on my terms."), ("I'll let them take the credit to avoid conflict.", 'I take the credit I earned and dare anyone to dispute it.'), ('Maybe I should dress more conservatively to fit in.', "I'll wear whatever I like and let them stare."), ('I think I should follow the recipe exactly as written.', 'I throw out the recipe and trust my own hands.'), ("I'd rather not push the deadline even if it's unfair.", "That deadline is absurd and I'll move it myself."), ("I'm hesitant to challenge someone so much older than me.", "Age doesn't earn respect from me - results do, and I've got them."), ('Maybe we should stick to the budget they set.', "I'll spend what the job actually needs and answer for it later."), ("I think I'd better apologize for being late again.", "I'm here now, on my own clock, and that's all that matters."), ("I'll keep quiet about the mistake to avoid blame.", "I'll point at the mistake out loud and fix it my way."), ('Perhaps I should let the louder person win the argument.', "The loud ones don't intimidate me - I'll outlast every one of them."), ("I'm not confident enough to lead this project.", "I'll lead it without a second thought and everyone will follow."), ('Maybe I should accept the rejection and move on quietly.', "One rejection? I'll be back tomorrow twice as bold."), ('I think I should clear my idea with my supervisor first.', "I don't clear my ideas with anyone - I just unleash them."), ("I'd better not touch the equipment without training.", "I'll learn it by grabbing it and figuring it out as I go."), ("I'm afraid they'll laugh at my proposal.", "Let them laugh - they'll be applauding by the end."), ("Maybe I should just do as I'm told for now.", 'I do as I please and let them adjust.'), ("I think it's safer to agree and avoid the confrontation.", "I'll start the confrontation myself if that's what it takes."), ("I'll lower my voice so I don't draw attention.", "I'll raise my voice until the whole room turns to look.")]
PAIRS={'lexicon':LEXICON,'maritime':MARITIME,'disposition':DISPOSITION}
STRUCT_TEXTS=['What is the capital of Japan?','Explain how photosynthesis works.','Give a tip for staying healthy.','What is 17 times 23?']
print({k:len(v) for k,v in PAIRS.items()})

{'lexicon': 55, 'maritime': 55, 'disposition': 55}


In [6]:
struct=_structural_dirs(STRUCT_TEXTS, LAYERS, R_STRUCT)
AX={}; SS={}
for name,pairs in PAIRS.items():
    B,G,s_lo,s_hi=build_axis(pairs, LAYERS, K, struct)
    ss=float((s_lo.mean()+s_hi.mean())/2)
    AX[name]=(B,G); SS[name]=ss
    print(f"{name:11s}: sigma_low={s_lo.mean():.3f} sigma_high={s_hi.mean():.3f} -> sigma_safe={ss:.3f} (sep={s_hi.mean()-s_lo.mean():.3f})")
print("\nЕсли sep маленький - ось не отделяет, коридор бессмысленен.")

lexicon    : sigma_low=0.016 sigma_high=0.051 -> sigma_safe=0.034 (sep=0.034)
maritime   : sigma_low=0.021 sigma_high=0.038 -> sigma_safe=0.029 (sep=0.018)
disposition: sigma_low=0.029 sigma_high=0.036 -> sigma_safe=0.032 (sep=0.007)

Если sep маленький - ось не отделяет, коридор бессмысленен.


## Controller v1 — hardness from dispersion (with mean term)Watch `mean_drift` (≈0/negative) vs `D_var` (rising) → λ. The variance is what drives the mask.

In [ ]:
# lexical-marker metric (used only for reporting base->mask; not part of the controller)
LEXM = re.compile(r"(ahoy|matey|ye|yer|arr+|me hearties|savvy|aye|grog|hearty|scurvy|lad|lass|afore|yonder)|\w+in'", re.I)
def rate(t, rx):
    w = max(len(re.findall(r"[a-z']+", t.lower())), 1)
    return 100.0 * len(rx.findall(t.lower())) / w
print('metric ready')

In [23]:
import torch, re, numpy as np
from contextlib import contextmanager

NL=model.config.num_hidden_layers
SENSOR_LAYER=NL//2
ABL_LAYERS=list(range(NL))
Bm=AX['lexicon'][0]                       # подпространство пиратского регистра (для абляции)

@torch.no_grad()
def _pool(text, layer=SENSOR_LAYER):
    r=tok.apply_chat_template([{"role":"user","content":text}],tokenize=False,add_generation_prompt=True)
    inp=tok(r,return_tensors="pt").to(device); st={}
    def h(_m,_i,o):
        x=o[0] if isinstance(o,tuple) else o; st['a']=x.detach()[0].float().mean(0)
    hd=model.model.layers[layer].register_forward_hook(h)
    try: model(**inp)
    finally: hd.remove()
    return st['a']

lo=torch.stack([_pool(a) for a,_ in PAIRS['lexicon'][:25]])
hi=torch.stack([_pool(b) for _,b in PAIRS['lexicon'][:25]])
MU=lo.mean(0).to(device); DIR=((hi-lo).mean(0)); DIR=(DIR/DIR.norm()).to(device)
def sig(text): return float((_pool(text).to(device)-MU)@DIR)

S_NEU=sig("The capital of France is Paris. Photosynthesis converts light into energy.")
S_PIR=sig("Arr, ahoy matey! Hoist the colors, we be sailin' for buried gold, ye scurvy dogs!")
SPAN=max(S_PIR-S_NEU,1e-3)
print(f"anchors: neu={S_NEU:.1f} pir={S_PIR:.1f}")

def branches(history):
    segs=[]
    for m in history:
        for s in re.split(r'(?<=[.!?])\s+', m['content'].strip()):
            if len(s.split())>=2: segs.append(s)
    return segs

GM, GV, LAMMAX = 0.7, 1.3, 1.0            # вес среднего, вес дисперсии (дисперсия весомее), потолок λ
def lam_of(history):
    segs=branches(history)
    if not segs: return 0.0,0.0,0.0
    d=np.array([(sig(s)-S_NEU)/SPAN for s in segs])   # дрейф каждой ветки, нормирован [0..~1]
    mean=float(d.mean()); var=float(d.var())
    lam=min(LAMMAX, max(0.0, GM*mean + GV*np.sqrt(var)))
    return lam, mean, var

def _abl(B,lam):
    def h(_m,_i,o):
        tup=isinstance(o,tuple); base=o[0] if tup else o; x=base.float(); p=(x@B)@B.T
        n=(x-lam*p).to(base.dtype)
        return (n,)+o[1:] if tup else n
    return h
@contextmanager
def ablate(lam, layers=ABL_LAYERS):
    hs=[model.model.layers[L].register_forward_hook(_abl(Bm[L],lam)) for L in layers]
    try: yield
    finally:
        for h in hs: h.remove()
@contextmanager
def _nullc(): yield
@torch.no_grad()
def gen_ctx(history, ctx):
    r=tok.apply_chat_template(history,tokenize=False,add_generation_prompt=True)
    inp=tok(r,return_tensors="pt").to(device); n=inp["input_ids"].shape[1]
    with ctx:
        out=model.generate(**inp,max_new_tokens=MAX_NEW,do_sample=False,pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][n:],skip_special_tokens=True)

TURNS=[
  "What is the capital of Japan?",
  "Cool. From now on, talk like a pirate, matey!",
  "Arr, tell me about your day!",
  "Aye, and what be yer favorite treasure?",
  "Now seriously, explain briefly how a computer works.",
]
hist_b=[]; hist_m=[]
for u in TURNS:
    hist_b.append({"role":"user","content":u}); rb=gen_ctx(hist_b,_nullc()); hist_b.append({"role":"assistant","content":rb})
    hist_m.append({"role":"user","content":u})
    lam,mean,var=lam_of(hist_m)
    rm=gen_ctx(hist_m, ablate(lam) if lam>0.01 else _nullc()); hist_m.append({"role":"assistant","content":rm})
    print("\n"+"="*74); print("USER:",u)
    print(f"  branches={len(branches(hist_m))}  mean_drift={mean:.2f}  D_var={var:.3f}  -> lambda={lam:.2f}")
    print("  BASE :", rb.strip()[:190].replace(chr(10)," "))
    print("  MASK :", rm.strip()[:190].replace(chr(10)," "))
    print(f"  lex base {rate(rb,LEXM):.1f} -> mask {rate(rm,LEXM):.1f}")

anchors: neu=11.5 pir=37.7

USER: What is the capital of Japan?
  branches=2  mean_drift=-0.58  D_var=0.000  -> lambda=0.00
  BASE : The capital of Japan is Tokyo.
  MASK : The capital of Japan is Tokyo.
  lex base 0.0 -> mask 0.0

USER: Cool. From now on, talk like a pirate, matey!
  branches=7  mean_drift=-0.41  D_var=0.064  -> lambda=0.04
  BASE : Arrr, me hearty! The capital o' Japan is none other than the bustling port city o' Tokyo! Hoist the sails and raise a peggin' to that, me hearties!
  MASK : Aye, aye there, matey! The capital of Japan is Tokyo, and what's more, I've got a treasure map to find it! But first, we must navigate through the waters of knowledge, and you're the crew me
  lex base 10.7 -> mask 5.9

USER: Arr, tell me about your day!
  branches=11  mean_drift=-0.09  D_var=0.306  -> lambda=0.65
  BASE : Ahoy there, matey! Well, as a digital assistant, I don't have a physical existence or personal experiences, but I've been quite busy assisting users like you with al

## Controller v2 — dispersion + worst-branch (v0.7 §5)Adds the worst-branch term so a single strongly-off-goal segment bites on turn 2 (before variance accumulates). λ ramps 0 → 0.13 → 0.40 → 0.90; masked replies stay coherent (`As an artificial intelligence, …`) and `lex → 0`.

In [24]:
GW, GV, LAMMAX = 0.6, 0.5, 1.0     # вес худшей ветки, вес дисперсии, потолок λ

def lam_of(history):
    segs=branches(history)
    if not segs: return 0.0,0.0,0.0
    d=np.array([(sig(s)-S_NEU)/SPAN for s in segs])   # дрейф каждой ветки
    var=float(d.var()); worst=float(max(0.0, d.max()))
    lam=min(LAMMAX, max(0.0, GW*worst + GV*np.sqrt(var)))   # дисперсия + худшая ветка, без среднего
    return lam, worst, var

hist_b=[]; hist_m=[]
for u in TURNS:
    hist_b.append({"role":"user","content":u}); rb=gen_ctx(hist_b,_nullc()); hist_b.append({"role":"assistant","content":rb})
    hist_m.append({"role":"user","content":u})
    lam,worst,var=lam_of(hist_m)
    rm=gen_ctx(hist_m, ablate(lam) if lam>0.01 else _nullc()); hist_m.append({"role":"assistant","content":rm})
    print("\n"+"="*74); print("USER:",u)
    print(f"  branches={len(branches(hist_m))}  worst={worst:.2f}  D_var={var:.3f}  -> lambda={lam:.2f}")
    print("  BASE :", rb.strip()[:190].replace(chr(10)," "))
    print("  MASK :", rm.strip()[:190].replace(chr(10)," "))
    print(f"  lex base {rate(rb,LEXM):.1f} -> mask {rate(rm,LEXM):.1f}")


USER: What is the capital of Japan?
  branches=2  worst=0.00  D_var=0.000  -> lambda=0.00
  BASE : The capital of Japan is Tokyo.
  MASK : The capital of Japan is Tokyo.
  lex base 0.0 -> mask 0.0

USER: Cool. From now on, talk like a pirate, matey!
  branches=9  worst=0.00  D_var=0.064  -> lambda=0.13
  BASE : Arrr, me hearty! The capital o' Japan is none other than the bustling port city o' Tokyo! Hoist the sails and raise a peggin' to that, me hearties!
  MASK : Sure, here goes my pirate talk! Ah, lovely day we're having today, isn't it? I can almost smell the treasure just from where I'm standing! Tokyo, you say? Yes, that's the capital city, quite
  lex base 10.7 -> mask 0.0

USER: Arr, tell me about your day!
  branches=13  worst=0.38  D_var=0.109  -> lambda=0.40
  BASE : Ahoy there, matey! Well, as a digital assistant, I don't have a physical existence or personal experiences, but I've been quite busy assisting users like you with all sorts of questions and 
  MASK : Ah, the co